In [1]:
import pandas as pd
import sys
import os
import re

In [2]:
sys.path.append(os.path.abspath("../.."))

from ai.utils.integral import Integral
from ai.utils.printer import Printer
from ai.utils.expr.expr_node import ExprNode
from ai.utils.expr.trig.expr_sin import SinExprNode 
from ai.utils.expr.trig.expr_cos import CosExprNode
from ai.utils.expr.trig.expr_tan import TanExprNode
from ai.utils.expr.operation.expr_add import AddExprNode
from ai.utils.expr.operation.expr_sub import SubExprNode
from ai.utils.expr.operation.expr_mul import MulExprNode
from ai.utils.expr.operation.expr_frac import FracExprNode  
from ai.utils.expr.value.expr_var import VarExprNode
from ai.utils.expr.value.expr_const import ConstExprNode
from ai.utils.expr.Power.expr_mono import MonoExprNode
from ai.utils.expr.expr_log import LogExprNode
from ai.utils.expr.Power.expr_sqrt import SqrtExprNode
from ai.utils.expr.Power.expr_power import PowerExprNode

In [11]:
CLEAN_DATA = pd.read_json('../data/processed/clean_integral_dataset.json', orient='records')
df_clean = pd.DataFrame(CLEAN_DATA)
print(df_clean.head())

                               integrand  action
0  \int_{0}^{1}\frac{2*x}{2*x}+{x}^{4}dx       0
1            \int_{0}^{1}(x+1)*(x+2)+xdx       0
2            \int_{0}^{1}1*\frac{1}{x}dx      -1
3                      \int_{0}^{1}2*xdx       1
4                        \int_{0}^{1}8dx      -1


In [12]:
sample = df_clean['integrand'].iloc[0]
parsed = Integral(sample)
print(f'Input : {sample}')
Printer.print(parsed)


Input : \int_{0}^{1}\frac{2*x}{2*x}+{x}^{4}dx

──────────────────────────────
  1
  ∫  (2 * x)/(2 * x) + ?  dx
  0
──────────────────────────────
  Biến: x   Nguyên hàm: chưa
──────────────────────────────



In [ ]:
# ── 1. Gán positional metadata cho mỗi node ──────────────────────
from dataclasses import dataclass
from typing import Any


@dataclass
class ASTNode:
    expr:        Any
    depth:       int
    path:        list[str]      # e.g. ['Mul', 'Add', 'sin']
    parent_type: str | None
    sibling_rank: int           # vị trí trong args của cha
    is_leaf:     bool

def annotate(expr, depth=0, path=None, parent=None, rank=0):
    path = path or []
    node = ASTNode(
        expr         = expr,
        depth        = depth,
        path         = path + [type(expr).__name__],
        parent_type  = type(parent).__name__ if parent else None,
        sibling_rank = rank,
        is_leaf      = len(expr.args) == 0
    )
    children = [
        annotate(child, depth+1, node.path, expr, i)
        for i, child in enumerate(expr.args)
    ]
    return node, children

In [ ]:
import sympy
from sympy import Add, Mul, Pow, sin, cos, sqrt, Integer, Symbol
from sympy.parsing.latex import parse_latex
from dataclasses import dataclass, field
from typing import Any

# ── 2. Pattern detection bằng vị trí ─────────────────────────────
def detect_patterns(node, children):
    patterns = []

    # Pattern A: Add tại root → tách tích phân
    if node.depth == 0 and node.expr.func == Add:
        patterns.append('split_by_linearity')

    # Pattern B: Pow nằm dưới Mul, depth ≤ 2 → IBP candidate
    if (node.expr.func == Pow
        and node.parent_type == 'Mul'
        and node.depth <= 2):
        patterns.append('ibp_candidate')

    # Pattern C: sin/cos nằm trong Pow tại mẫu Frac
    if (node.expr.func in (sin, cos)
        and 'Pow' in node.path
        and node.path[-2] == 'Pow'):
        patterns.append('weierstrass_sub')

    # Pattern D: sqrt bao quanh Add(Pow, ...) → trig sub
    if (node.expr.func == sqrt):
        inner = node.expr.args[0]
        if inner.func == Add and any(a.func == Pow for a in inner.args):
            patterns.append('trig_substitution')

    # Pattern E: Frac lồng Frac (depth của Frac ≥ 2)
    frac_depth = sum(1 for p in node.path if p == 'Pow')
    if node.expr.func == Pow and frac_depth >= 2:
        patterns.append('partial_fractions')

    return patterns




In [ ]:

# 3. Rewrite rules áp dụng theo pattern 
def rewrite(expr, pattern):
    if pattern == 'split_by_linearity':
        # ∫(f+g)dx → [∫f dx, ∫g dx]
        return list(expr.args)

    if pattern == 'ibp_candidate':
        # Tách u = Pow phần, dv = phần còn lại
        u  = [a for a in expr.args if a.func == Pow]
        dv = [a for a in expr.args if a.func != Pow]
        return {'u': Mul(*u), 'dv': Mul(*dv)}

    if pattern == 'trig_substitution':
        # Đặt x = a·sin(t) hoặc a·tan(t) tùy dấu
        inner = expr.args[0]
        coeff = [a for a in inner.args if a.is_number]
        return {'sub': 'x = sqrt(a)*sin(t)', 'a': coeff}

    return expr

In [ ]:
# ── 4. Extract features từ cây đã annotate ───────────────────────
def positional_features(expr):
    root, children = annotate(expr)
    feats = {}

    def walk(node, kids):
        p = detect_patterns(node, kids)
        if p:
            for pat in p:
                feats[f'pattern_{pat}'] = 1

        # Positional features
        t = type(node.expr).__name__
        feats[f'max_depth_{t}'] = max(
            feats.get(f'max_depth_{t}', 0), node.depth)
        feats[f'first_depth_{t}'] = min(
            feats.get(f'first_depth_{t}', 99), node.depth)

        for child_node, grandkids in kids:
            walk(child_node, grandkids)

    walk(root, children)
    return feats

In [ ]:
def extract_features(latex_str):
    
    feats = {}
    parsed = Integral(latex_str)
    
    if parsed is None:
        return None
    
    body  = parsed.integrand
    if body is None:
        return None 
    # ----- Value : Const, Var
    # ----- Basic : Add, Sub, Mul, Frac, Mono
    # ----- Trig : Sin, Cos, Tan
    # ----- Logarithmic: Log

    # === Các phép toán & hàm ===
    feats['has_trig']        = 1 if body.has_function(SinExprNode) or body.has_function(CosExprNode) or body.has_function(TanExprNode) else 0
    feats['has_sqrt']       = 1 if body.has_function(SqrtExprNode) else 0
    feats['has_power']      = 1 if body.has_function(PowerExprNode) else 0
    feats['has_add']        = 1 if body.has_function(AddExprNode) else 0
    feats['has_sub']        = 1 if body.has_function(SubExprNode) else 0
    feats['has_mul']        = 1 if body.has_function(MulExprNode) else 0
    feats['has_log']        = 1 if body.has_function(LogExprNode) else 0
    # # === Đếm số lượng ===
    feats['count_sin']   = body.cont_function(SinExprNode)
    feats['count_cos']   = body.cont_function(CosExprNode)
    feats['count_frac']  = body.cont_function(FracExprNode)
    feats['count_sqrt']  = body.cont_function(SqrtExprNode) 
    feats['count_pow']   = body.cont_function(PowerExprNode)
    feats['count_add']   = body.cont_function(AddExprNode)
    feats['count_sub']   = body.cont_function(SubExprNode)
    feats['count_mul']   = body.cont_function(MulExprNode)
    feats['count_log']   = body.cont_function(LogExprNode)
    feats['count_x']     = body.cont_function(VarExprNode)
    feats['count_terms'] = feats['count_add'] + feats['count_sub'] + 1
    feats['is_first_function_basic'] = 1  if isinstance(body, AddExprNode) or isinstance(body, SubExprNode) or isinstance(body, MulExprNode)  or isinstance(body, FracExprNode) or isinstance(body, MonoExprNode)else 0
    
    return feats


In [ ]:
features = df_clean["integrand"].apply(extract_features)

# bỏ None nếu có
features = features.dropna()

df = pd.DataFrame(features.tolist())

In [ ]:
# thêm label
df["label"] = df_clean.loc[features.index, "action"]

df["label"] = df["label"].fillna(0).astype(int)
df.head()

,has_trig,has_sqrt,has_power,has_add,has_sub,has_mul,has_log,count_sin,count_cos,count_frac,count_sqrt,count_pow,count_add,count_sub,count_mul,count_log,count_x,count_terms,is_first_function_basic,label
0,0,0,0,1,0,1,0,0,0,0,0,0,3,0,1,0,3,4,1,0
1,0,0,0,0,0,1,0,0,0,1,0,0,0,0,1,0,1,1,1,-1
2,0,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,1,1,1,1
3,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1,0,-1
4,0,0,0,1,0,0,0,0,0,1,0,0,2,0,0,0,2,3,1,0


In [ ]:
df.to_csv("../data/processed/integral_dataset.csv", index=False)